# AVISO velocity versus ESP reconstruction

Compare the native AVISO velocity field with the velocity reconstructed from the fitted ESP/DOPPIO parameters. The notebook automatically selects three long-lived anticyclonic eddies (AEs) and three long-lived cyclonic eddies (CEs), then shows early, middle, and late stages of each life.

Only the displayed AVISO field may have a constant far-field background removed. DOPPIO reconstruction always represents the fitted eddy component.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import xarray as xr

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for source in (
    PROJECT_ROOT / "src",
    PROJECT_ROOT.parent / "seacofs_eddy_dataset_modular" / "src",
):
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))

from aviso_eddy_dataset.grid import build_grid, native_velocity

ESP_ROOT = Path("/home/z5297792/ESP_zonodo")
if str(ESP_ROOT) not in sys.path:
    sys.path.insert(0, str(ESP_ROOT))
import functions as esp

## Controls

In [ ]:
DATA_PATH = Path(
    "/srv/scratch/z5297792/aviso_eddy_dataset/processed/eddy_dataset_processed.parquet"
)
N_EDDIES_PER_POLARITY = 3
LIFE_STAGES = {"Early": 0.2, "Middle": 0.5, "Late": 0.8}
HALF_WIDTH_RC = 2.25
MIN_HALF_WIDTH_KM = 80.0
REMOVE_BACKGROUND = True
BACKGROUND_OUTER_FRACTION = 0.72
QUIVER_STEP = 1
SAVE_FIGURES = False
FIGURE_DIR = PROJECT_ROOT / "figures" / "aviso_esp_reconstruction"
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## Load processed eddies and select cases

Selection uses the three longest valid tracks of each polarity. A valid row must contain the full fitted parameter set and a source NetCDF path.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
parameter_columns = ["xc", "yc", "q11", "q12", "q22", "Omega", "Rc"]
valid = df.dropna(subset=parameter_columns + ["source_file"]).copy()
source_exists = {value: Path(str(value)).exists() for value in valid["source_file"].unique()}
valid = valid.loc[valid["source_file"].map(source_exists)]

track_counts = (
    valid.groupby(["Cyc", "Eddy"])
    .agg(valid_days=("Day", "size"), first_date=("Date", "min"), last_date=("Date", "max"))
    .reset_index()
)
selected = (
    track_counts.sort_values(["Cyc", "valid_days"], ascending=[True, False])
    .groupby("Cyc", group_keys=False)
    .head(N_EDDIES_PER_POLARITY)
)
counts = selected.groupby("Cyc").size()
for polarity in ("AE", "CE"):
    if counts.get(polarity, 0) < N_EDDIES_PER_POLARITY:
        raise ValueError(f"Only {counts.get(polarity, 0)} valid {polarity} tracks are available.")
selected

In [ ]:
def rows_at_life_stages(table, eddy_id, stages=LIFE_STAGES):
    track = table.loc[table["Eddy"].eq(eddy_id)].sort_values("Day").reset_index(drop=True)
    if track.empty:
        raise ValueError(f"No valid rows for eddy {eddy_id}.")
    rows = {}
    for label, fraction in stages.items():
        index = int(round(fraction * (len(track) - 1)))
        rows[label] = track.iloc[index]
    return rows

cases = {
    polarity: {
        int(eddy_id): rows_at_life_stages(valid, int(eddy_id))
        for eddy_id in selected.loc[selected["Cyc"].eq(polarity), "Eddy"]
    }
    for polarity in ("AE", "CE")
}
pd.DataFrame(
    [
        {"Cyc": polarity, "Eddy": eddy_id, "Stage": stage, "Day": int(row.Day), "Date": row.Date}
        for polarity, eddies in cases.items()
        for eddy_id, stages in eddies.items()
        for stage, row in stages.items()
    ]
)

## Load native AVISO and reconstruct the ESP field

In [ ]:
def time_index(dataset, date):
    target = np.datetime64(pd.Timestamp(date).to_datetime64())
    times = dataset["time"].values.astype("datetime64[ns]")
    index = int(np.argmin(np.abs(times - target)))
    if pd.Timestamp(times[index]).normalize() != pd.Timestamp(date).normalize():
        raise KeyError(f"Date {pd.Timestamp(date).date()} is absent from the source file.")
    return index


def remove_constant_background(u, v, fraction=BACKGROUND_OUTER_FRACTION):
    nx, ny = u.shape
    xi, yi = np.ogrid[-1:1:complex(nx), -1:1:complex(ny)]
    outer = np.maximum(np.abs(xi), np.abs(yi)) >= fraction
    background_u = np.nanmedian(u[outer])
    background_v = np.nanmedian(v[outer])
    return u - background_u, v - background_v, (background_u, background_v)


def comparison_for_row(row):
    source = Path(str(row.source_file))
    with xr.open_dataset(source, decode_times=True) as dataset:
        grid = build_grid(dataset["longitude"].values, dataset["latitude"].values)
        t = time_index(dataset, row.Date)
        u_original = native_velocity(dataset["ugos"].isel(time=t), grid)
        v_original = native_velocity(dataset["vgos"].isel(time=t), grid)

    half_width = max(MIN_HALF_WIDTH_KM, HALF_WIDTH_RC * float(row.Rc))
    ii = np.flatnonzero((grid.x_grid >= row.xc - half_width) & (grid.x_grid <= row.xc + half_width))
    jj = np.flatnonzero((grid.y_grid >= row.yc - half_width) & (grid.y_grid <= row.yc + half_width))
    if len(ii) < 5 or len(jj) < 5:
        raise ValueError(f"Comparison box is too small for eddy {int(row.Eddy)}, day {int(row.Day)}.")
    index = np.ix_(ii, jj)
    X, Y = grid.X_grid[index], grid.Y_grid[index]
    u_original, v_original = u_original[index], v_original[index]
    background = (0.0, 0.0)
    if REMOVE_BACKGROUND:
        u_original, v_original, background = remove_constant_background(u_original, v_original)

    Q = np.array([[row.q11, row.q12], [row.q12, row.q22]], dtype=float)
    u_esp, v_esp = esp.model_uv_at_xy(
        X * 1e3, Y * 1e3, row.xc * 1e3, row.yc * 1e3, Q, row.Omega, row.Rc * 1e3
    )
    valid_mask = np.isfinite(u_original) & np.isfinite(v_original)
    u_esp = np.where(valid_mask, u_esp, np.nan)
    v_esp = np.where(valid_mask, v_esp, np.nan)
    return {
        "X": X, "Y": Y, "u_original": u_original, "v_original": v_original,
        "u_esp": u_esp, "v_esp": v_esp, "background": background, "row": row,
    }


def vector_skill(comparison):
    uo, vo = comparison["u_original"], comparison["v_original"]
    ue, ve = comparison["u_esp"], comparison["v_esp"]
    good = np.isfinite(uo) & np.isfinite(vo) & np.isfinite(ue) & np.isfinite(ve)
    if good.sum() < 3:
        return np.nan, np.nan
    rmse = np.sqrt(np.mean((ue[good] - uo[good]) ** 2 + (ve[good] - vo[good]) ** 2))
    observed_speed = np.hypot(uo[good], vo[good])
    reconstructed_speed = np.hypot(ue[good], ve[good])
    correlation = pearsonr(observed_speed, reconstructed_speed)[0] if np.std(observed_speed) and np.std(reconstructed_speed) else np.nan
    return rmse, correlation

In [ ]:
comparisons = {
    polarity: {
        eddy_id: {stage: comparison_for_row(row) for stage, row in stages.items()}
        for eddy_id, stages in eddies.items()
    }
    for polarity, eddies in cases.items()
}
print("Loaded native AVISO fields and reconstructed all 18 eddy stages.")

## Side-by-side comparisons

Each row is one eddy. Columns are paired as original/reconstruction for early, middle, and late life. Colour limits are shared within each pair.

In [ ]:
def plot_polarity(polarity, comparisons_for_polarity):
    eddy_ids = list(comparisons_for_polarity)
    stage_names = list(LIFE_STAGES)
    fig, axes = plt.subplots(
        len(eddy_ids), 2 * len(stage_names), figsize=(19, 4.8 * len(eddy_ids)),
        constrained_layout=True, squeeze=False,
    )
    polarity_name = "Anticyclonic" if polarity == "AE" else "Cyclonic"

    for row_index, eddy_id in enumerate(eddy_ids):
        for stage_index, stage in enumerate(stage_names):
            comparison = comparisons_for_polarity[eddy_id][stage]
            row = comparison["row"]
            X, Y = comparison["X"], comparison["Y"]
            uo, vo = comparison["u_original"], comparison["v_original"]
            ue, ve = comparison["u_esp"], comparison["v_esp"]
            speeds = (np.hypot(uo, vo), np.hypot(ue, ve))
            vmax = np.nanpercentile(np.concatenate([field.ravel() for field in speeds]), 99)
            rmse, correlation = vector_skill(comparison)

            for pair_index, (u, v, speed, label) in enumerate(
                [(uo, vo, speeds[0], "AVISO anomaly" if REMOVE_BACKGROUND else "AVISO original"),
                 (ue, ve, speeds[1], "ESP reconstruction")]
            ):
                ax = axes[row_index, 2 * stage_index + pair_index]
                mesh = ax.pcolormesh(X, Y, speed, cmap="magma", vmin=0, vmax=vmax, shading="auto")
                step = max(1, QUIVER_STEP)
                ax.quiver(
                    X[::step, ::step], Y[::step, ::step],
                    u[::step, ::step], v[::step, ::step],
                    color="white", alpha=0.8, pivot="mid", width=0.004,
                )
                ax.scatter(row.xc, row.yc, color="cyan", edgecolor="black", s=35, zorder=3)
                ax.set_aspect("equal")
                ax.set_xlabel("x (km)")
                if pair_index == 0:
                    ax.set_ylabel(f"Eddy {eddy_id}\ny (km)")
                title = f"{stage}: {label}\n{row.Date:%Y-%m-%d}"
                if pair_index == 1:
                    title += f" | RMSE={rmse:.3f}, r={correlation:.2f}"
                ax.set_title(title, fontsize=9)
                fig.colorbar(mesh, ax=ax, label="Speed (m s$^{-1}$)", shrink=0.78)

    background_note = "constant far-field background removed" if REMOVE_BACKGROUND else "raw ugos/vgos"
    fig.suptitle(
        f"{polarity_name} eddies ({polarity}): native AVISO versus ESP reconstruction\n"
        f"Original panels: {background_note}", fontsize=15, fontweight="bold",
    )
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / f"{polarity.lower()}_original_vs_esp.png", dpi=300, bbox_inches="tight")
    plt.show()


plot_polarity("AE", comparisons["AE"])
plot_polarity("CE", comparisons["CE"])

## Skill summary

In [ ]:
skill_rows = []
for polarity, eddies in comparisons.items():
    for eddy_id, stages in eddies.items():
        for stage, comparison in stages.items():
            rmse, correlation = vector_skill(comparison)
            row = comparison["row"]
            skill_rows.append(
                {
                    "Cyc": polarity, "Eddy": eddy_id, "Stage": stage,
                    "Date": row.Date, "Day": int(row.Day),
                    "Rc_km": row.Rc, "R_km": row.R,
                    "Vector_RMSE_m_s": rmse, "Speed_correlation": correlation,
                    "background_u_m_s": comparison["background"][0],
                    "background_v_m_s": comparison["background"][1],
                }
            )
skill = pd.DataFrame(skill_rows)
skill.style.format(
    {"Rc_km": "{:.1f}", "R_km": "{:.1f}", "Vector_RMSE_m_s": "{:.3f}",
     "Speed_correlation": "{:.2f}", "background_u_m_s": "{:.3f}",
     "background_v_m_s": "{:.3f}"}
)